# 03 - Training (GridSearchCV, k=10)

Trains 4 models with `GridSearchCV` (10-fold stratified on the **training set**, `scoring='f1_macro'`): Random Forest, SVM-RBF, Gradient Boosting, and dense MLP (`MLPClassifier`). All models receive scaled features (uniform inference path; scaling trees is harmless). Saves the best estimators to `models/` via joblib.

In [1]:
import sys
from pathlib import Path

_p = Path.cwd()
for ROOT in [_p, *_p.parents]:
    if (ROOT / "requirements.txt").exists():
        break
sys.path.insert(0, str(ROOT / "src"))

import warnings

import joblib
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

from preprocessing import load_feature_table, split_features_metadata

SPLITS_DIR = ROOT / "data" / "splits"
MODELS_DIR = ROOT / "models"
OUT_DIR = ROOT / "outputs" / "pilot" / "modeling"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
CV = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)

In [2]:
train = load_feature_table(SPLITS_DIR / "train.parquet")
X_train, y_train, _ = split_features_metadata(train)
scaler = joblib.load(MODELS_DIR / "scaler.pkl")
X_train_s = scaler.transform(X_train)
print("train:", X_train_s.shape, "| classes:", y_train.value_counts().to_dict())

train: (840, 648) | classes: {0: 420, 1: 420}


In [3]:
search_space = {
    "random_forest": (
        RandomForestClassifier(random_state=RANDOM_SEED),
        {"n_estimators": [200, 400], "max_depth": [None, 10, 20]},
    ),
    "svm": (
        SVC(kernel="rbf", random_state=RANDOM_SEED),
        {"C": [1, 10, 100], "gamma": ["scale", 0.01, 0.001]},
    ),
    "gradient_boosting": (
        GradientBoostingClassifier(random_state=RANDOM_SEED),
        {"n_estimators": [100, 200], "max_depth": [2, 3], "learning_rate": [0.05, 0.1]},
    ),
    "mlp": (
        MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_SEED),
        {"hidden_layer_sizes": [(128, 64), (128, 64, 32)], "alpha": [1e-4, 1e-3]},
    ),
}

In [4]:
rows = []
for name, (estimator, grid) in search_space.items():
    search = GridSearchCV(estimator, grid, cv=CV, scoring="f1_macro", n_jobs=-1, refit=True)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        search.fit(X_train_s, y_train)
    joblib.dump(search.best_estimator_, MODELS_DIR / f"{name}.pkl")
    rows.append({"model": name, "cv_f1_macro": search.best_score_,
                 "best_params": search.best_params_})
    print(f"{name}: cv f1_macro={search.best_score_:.4f} | {search.best_params_}")

cv_results = pd.DataFrame(rows).sort_values("cv_f1_macro", ascending=False)
cv_results.to_csv(OUT_DIR / "cv_results.csv", index=False)
cv_results

random_forest: cv f1_macro=1.0000 | {'max_depth': None, 'n_estimators': 200}
svm: cv f1_macro=1.0000 | {'C': 1, 'gamma': 'scale'}
gradient_boosting: cv f1_macro=0.9976 | {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 200}
mlp: cv f1_macro=0.9988 | {'alpha': 0.0001, 'hidden_layer_sizes': (128, 64)}


,model,cv_f1_macro,best_params
0,random_forest,1.000000,"{'max_depth': None, 'n_estimators': 200}"
1,svm,1.000000,"{'C': 1, 'gamma': 'scale'}"
3,mlp,0.998809,"{'alpha': 0.0001, 'hidden_layer_sizes': (128, ..."
2,gradient_boosting,0.997619,"{'learning_rate': 0.05, 'max_depth': 2, 'n_est..."
